# Combination Model: Multi-Headline FinBERT Aggregation → LinearSVC

Same multi-headline sentiment aggregation as `combination_multisent.ipynb`,  
but uses **LinearSVC** (proven best on this dataset) instead of Transformer.  
**11 sentiment + price features.** Target: intraday direction (close ≥ open, |move| > 0.2%).

In [1]:
# Colab setup
# torch / transformers pre-installed on Colab, nothing extra needed

In [2]:
import os, gc, shutil
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from google.colab import files

WORK_DIR   = "/content"
DATA_FILE  = f"{WORK_DIR}/sp500_headlines_with_prices sep ~.csv"
CACHE_FILE = f"{WORK_DIR}/finbert_probs_multisent.npy"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
np.random.seed(42)

Using device: cuda


In [3]:
# Upload data file
TARGET_NAME = "sp500_headlines_with_prices sep ~.csv"

if not os.path.exists(DATA_FILE):
    print(f"Please upload: {TARGET_NAME}")
    uploaded = files.upload()
    for filename in uploaded.keys():
        shutil.move(filename, os.path.join(WORK_DIR, TARGET_NAME))
        print(f"Saved to {WORK_DIR}/{TARGET_NAME}")
else:
    print(f"{TARGET_NAME} already exists, skipping upload.")

Please upload: sp500_headlines_with_prices sep ~.csv


Saving sp500_headlines_with_prices sep ~.csv to sp500_headlines_with_prices sep ~.csv
Saved to /content/sp500_headlines_with_prices sep ~.csv


In [4]:
# 1. Load data & target — filter out near-flat days
df = pd.read_csv(DATA_FILE)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

df["target"] = (df["close_price"] >= df["open_price"]).astype(int)

MOVE_THRESHOLD = 0.002
df["daily_move"] = (df["close_price"] - df["open_price"]).abs() / df["open_price"]
before = len(df)
df = df[df["daily_move"] > MOVE_THRESHOLD].reset_index(drop=True)

print(f"Total rows: {before} → {len(df)} after removing near-flat days (<{MOVE_THRESHOLD:.1%} move)")
print(f"Target distribution: {df['target'].value_counts().to_dict()}")

Total rows: 3507 → 2572 after removing near-flat days (<0.2% move)
Target distribution: {1: 1425, 0: 1147}


In [5]:
# 2. FinBERT — score each headline individually, aggregate per day
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
finbert   = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert").to(device)
finbert.eval()

def get_finbert_probs(texts, batch_size=64):
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch  = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt",
                           truncation=True, max_length=128, padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = finbert(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        all_probs.append(probs.cpu().numpy())
    return np.vstack(all_probs)

records = []
for idx, row in df.iterrows():
    for h in str(row["combined_headline"]).split(" ~ "):
        h = h.strip()
        if h:
            records.append({"day_idx": idx, "headline": h})
headlines_df = pd.DataFrame(records)
print(f"Total individual headlines: {len(headlines_df)}  (avg {len(headlines_df)/len(df):.1f}/day)")

if os.path.exists(CACHE_FILE):
    print("Loading cached per-headline FinBERT probs...")
    flat_probs = np.load(CACHE_FILE)
    if len(flat_probs) != len(headlines_df):
        print(f"Cache size mismatch ({len(flat_probs)} vs {len(headlines_df)}), recomputing...")
        raise ValueError("recompute")
else:
    print("Running FinBERT on individual headlines (~5 min on T4)...")
    flat_probs = get_finbert_probs(headlines_df["headline"].tolist())
    np.save(CACHE_FILE, flat_probs)
    print(f"Saved to {CACHE_FILE}")

headlines_df["prob_pos"] = flat_probs[:, 0]
headlines_df["prob_neg"] = flat_probs[:, 1]
headlines_df["prob_neu"] = flat_probs[:, 2]

agg = headlines_df.groupby("day_idx").agg(
    sent_pos_mean = ("prob_pos", "mean"),
    sent_neg_mean = ("prob_neg", "mean"),
    sent_neu_mean = ("prob_neu", "mean"),
    sent_pos_max  = ("prob_pos", "max"),
    sent_neg_max  = ("prob_neg", "max"),
    sent_variance = ("prob_pos", "var"),
    n_headlines   = ("prob_pos", "count"),
).fillna(0)

df = df.join(agg)

del finbert, tokenizer
torch.cuda.empty_cache() if torch.cuda.is_available() else None
gc.collect()
print("Done. Model freed from memory.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total individual headlines: 14297  (avg 5.6/day)
Running FinBERT on individual headlines (~5 min on T4)...


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Saved to /content/finbert_probs_multisent.npy
Done. Model freed from memory.


In [6]:
# 3. Feature engineering
df["gap_open"]    = (df["open_price"] - df["close_price"].shift(1)) / df["close_price"].shift(1)
df["_ma5"]        = df["close_price"].rolling(5).mean()
df["_vol_ma5"]    = df["volume"].rolling(5).mean()

def compute_rsi(series, period=14):
    delta    = series.diff()
    gain     = delta.clip(lower=0)
    loss     = (-delta).clip(lower=0)
    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()
    rs  = avg_gain / (avg_loss + 1e-9)
    return (100 - 100 / (1 + rs)) / 100.0

df["_rsi14"]         = compute_rsi(df["close_price"], 14)
df["prev_ma5_ratio"] = df["close_price"].shift(1) / df["_ma5"].shift(1)
df["prev_rsi14"]     = df["_rsi14"].shift(1)
df["prev_vol_ratio"] = df["volume"].shift(1) / df["_vol_ma5"].shift(1)

FEATURE_COLS = [
    "sent_pos_mean", "sent_neg_mean", "sent_neu_mean",
    "sent_pos_max",  "sent_neg_max",
    "sent_variance", "n_headlines",
    "gap_open",
    "prev_ma5_ratio", "prev_rsi14", "prev_vol_ratio",
]

df = df.dropna(subset=FEATURE_COLS + ["target"]).reset_index(drop=True)
print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"Rows after dropna: {len(df)}")

Features (11): ['sent_pos_mean', 'sent_neg_mean', 'sent_neu_mean', 'sent_pos_max', 'sent_neg_max', 'sent_variance', 'n_headlines', 'gap_open', 'prev_ma5_ratio', 'prev_rsi14', 'prev_vol_ratio']
Rows after dropna: 2557


In [7]:
# 4. Train / test split (chronological) + scale
split_index = int(len(df) * 0.8)

X_train = df[FEATURE_COLS].iloc[:split_index].values
y_train = df["target"].iloc[:split_index].values
X_test  = df[FEATURE_COLS].iloc[split_index:].values
y_test  = df["target"].iloc[split_index:].values

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Test target dist: {dict(zip(*np.unique(y_test, return_counts=True)))}")

Train: (2045, 11) | Test: (512, 11)
Test target dist: {np.int64(0): np.int64(238), np.int64(1): np.int64(274)}


In [11]:
# 5. LinearSVC with multi-headline features
svc = CalibratedClassifierCV(LinearSVC(C=1.0, max_iter=5000, random_state=42, class_weight="balanced"))
svc.fit(X_train, y_train)

svc_probs = svc.predict_proba(X_test)[:, 1]
svc_preds = svc.predict(X_test)

print("=== LinearSVC (MultiSent, 11 features, balanced) ===")
print(classification_report(y_test, svc_preds))
print("Accuracy:", round(accuracy_score(y_test, svc_preds), 4))
print("ROC-AUC :", round(roc_auc_score(y_test, svc_probs), 4))

=== LinearSVC (MultiSent, 11 features, balanced) ===
              precision    recall  f1-score   support

           0       0.75      0.01      0.02       238
           1       0.54      1.00      0.70       274

    accuracy                           0.54       512
   macro avg       0.64      0.50      0.36       512
weighted avg       0.64      0.54      0.39       512

Accuracy: 0.5391
ROC-AUC : 0.6454


In [12]:
# 6. Final comparison
majority = y_test.sum() / len(y_test)
print("======================================================")
print(f"  Random baseline (always up)                : {majority:.1%}")
print(f"  StockPredict   FinBERT→LinearSVC           : 64.1%")
print(f"  Combination    LinearSVC (7 features)      : 64.2%")
print(f"  MultiSent      LinearSVC (11 feat+filter)  : {accuracy_score(y_test, svc_preds):.1%}")
print("======================================================")

  Random baseline (always up)                : 53.5%
  StockPredict   FinBERT→LinearSVC           : 64.1%
  Combination    LinearSVC (7 features)      : 64.2%
  MultiSent      LinearSVC (11 feat+filter)  : 53.9%


In [13]:
# 7. LinearSVC coefficients
coef = pd.Series(
    svc.calibrated_classifiers_[0].estimator.coef_[0],
    index=FEATURE_COLS
).sort_values(ascending=False)
print("--- LinearSVC Coefficients (+ = bullish signal) ---")
print(coef.to_string())

--- LinearSVC Coefficients (+ = bullish signal) ---
gap_open          0.244286
sent_pos_max      0.048624
sent_pos_mean     0.044315
n_headlines       0.030428
prev_vol_ratio    0.007591
sent_neg_mean     0.003796
sent_variance    -0.010546
prev_ma5_ratio   -0.015340
prev_rsi14       -0.024241
sent_neu_mean    -0.033423
sent_neg_max     -0.111543


In [14]:
from sklearn.metrics import confusion_matrix

tn, fp, fn, tp = confusion_matrix(y_test, svc_preds).ravel()

print(f"Accuracy  : {accuracy_score(y_test, svc_preds):.4f}")
print(f"Precision : {tp / (tp + fp):.4f}")
print(f"Recall    : {tp / (tp + fn):.4f}")
print(f"F1-score  : {2*tp / (2*tp + fp + fn):.4f}")
print(f"TP: {tp}  TN: {tn}  FP: {fp}  FN: {fn}")

Accuracy  : 0.5391
Precision : 0.5374
Recall    : 0.9964
F1-score  : 0.6982
TP: 273  TN: 3  FP: 235  FN: 1
